# Claude Agent SDK: how the agent loop works

Every agent you build with the SDK runs the same engine. Understanding it tells you **why your agent did what it did**, how to **cap its cost**, how to **keep it safe**, and how to **debug it**.

**What you will learn (and see running):**
1. The loop: prompt → evaluate → tool calls → results → repeat → final answer
2. Turns and the five message types, with a live trace
3. Tools, and the three knobs that control them (`allowed_tools`, `disallowed_tools`, `permission_mode`)
4. Limits: `max_turns`, `max_budget_usd`, `effort`
5. The context window and automatic compaction
6. Reading the `ResultMessage`
7. Hooks: code that runs *around* tool calls

> The agent cells call the API and cost a small number of tokens. Every demo works inside its own folder, so nothing else on your machine is touched.

## The big picture

```
                      ┌──────────────────────────────────────────────┐
  your prompt ──────▶ │  1. Claude evaluates the situation           │
                      │        │                                     │
                      │        ├── wants tools? ──▶ 2. SDK runs them  │
                      │        │                      │              │
                      │        │      results fed back ◀─┘           │
                      │        │      (that round trip = one TURN)   │
                      │        │                                     │
                      │        └── no tools ──▶ final answer         │
                      └──────────────────────────────────────────────┘
                                                   │
                                          ResultMessage (cost, usage, session id)
```

1. **Receive prompt**: Claude gets your prompt + system prompt + tool definitions + history. The SDK yields a `SystemMessage` (subtype `init`).
2. **Evaluate and respond**: Claude answers with text, tool-call requests, or both. Each content block arrives as an `AssistantMessage`.
3. **Execute tools**: the SDK runs the requested tools and feeds results back. Hooks can intercept here.
4. **Repeat** steps 2-3. One full cycle is a **turn**.
5. **Return result**: a text-only `AssistantMessage`, then a `ResultMessage` with cost, usage and session id.

A simple question can finish in 1-2 turns. A big task can chain dozens.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv 

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("agent-loop-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. A tiny project with a failing test

To watch the loop work, the agent needs a real task that **requires several turns**: run the tests, read the code, fix it, run the tests again.

`calculator.py` has a bug (`add` subtracts) and `test_calculator.py` will fail until it is fixed.

In [ ]:
BUGGY_CALCULATOR = """def add(a, b):
    return a - b


def multiply(a, b):
    return a * b
"""

TESTS = """import unittest
from calculator import add, multiply


class TestCalculator(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(2, 3), 5)

    def test_multiply(self):
        self.assertEqual(multiply(2, 3), 6)


if __name__ == "__main__":
    unittest.main()
"""

def reset_demo():
    """Put the demo folder back in its original (buggy) state."""
    (DEMO_DIR / "calculator.py").write_text(BUGGY_CALCULATOR)
    (DEMO_DIR / "test_calculator.py").write_text(TESTS)
    print("Demo reset in", DEMO_DIR)

reset_demo()

In [ ]:
import subprocess, sys

def run_tests():
    """Run the unit tests ourselves (no agent involved) and show the outcome."""
    p = subprocess.run([sys.executable, "-m", "unittest", "-q"], cwd=DEMO_DIR,
                       capture_output=True, text=True)
    print(p.stderr.strip() or p.stdout.strip())
    print("=> PASS" if p.returncode == 0 else "=> FAIL")

run_tests()

## 2. Watch the loop: a message-by-message trace

`query()` returns an **async iterator**. Each thing that happens inside the loop is yielded to you as a message. The helper below prints them in a readable way and marks where each **turn** ends.

Message types you will see:

| Type | Meaning |
|---|---|
| `SystemMessage` | Lifecycle events. `subtype="init"` first; `"compact_boundary"` after compaction |
| `AssistantMessage` | One content block of Claude's reply: a `TextBlock` or a `ToolUseBlock` |
| `UserMessage` | Tool **results** sent back to Claude (plus any input you stream mid-loop) |
| `StreamEvent` | Raw streaming deltas, only if `include_partial_messages=True` |
| `ResultMessage` | The end of the loop: result text, usage, cost, session id |

In [ ]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, HookMatcher,
    AssistantMessage, UserMessage, SystemMessage, ResultMessage,
    TextBlock, ToolUseBlock, ToolResultBlock,
)

def short(value, n=100):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"


async def trace(prompt, options):
    """Run one agent loop, print every message, and return all messages."""
    messages, turn = [], 0
    try:
        async for m in query(prompt=prompt, options=options):
            messages.append(m)

            if isinstance(m, SystemMessage):
                extra = f" session={m.data.get('session_id')}" if m.subtype == "init" else ""
                print(f"[system:{m.subtype}]{extra}")

            elif isinstance(m, AssistantMessage):
                for block in m.content:
                    if isinstance(block, TextBlock):
                        print(f"  assistant says : {short(block.text)}")
                    elif isinstance(block, ToolUseBlock):
                        print(f"  assistant calls: {block.name}({short(block.input, 80)})")

            elif isinstance(m, UserMessage):
                blocks = m.content if isinstance(m.content, list) else []
                for block in blocks:
                    if isinstance(block, ToolResultBlock):
                        flag = " (error)" if block.is_error else ""
                        print(f"  tool result{flag} : {short(block.content, 80)}")
                if blocks:
                    turn += 1
                    print(f"  ── end of turn {turn} ──")

            elif isinstance(m, ResultMessage):
                cost = f"${m.total_cost_usd:.4f}" if m.total_cost_usd is not None else "n/a"
                print(f"[result:{m.subtype}] turns={m.num_turns} cost={cost} stop_reason={m.stop_reason}")
    except Exception as error:
        # A single-shot query() raises AFTER yielding an error result (e.g. max turns).
        print(f"(query raised after finishing: {short(error, 120)})")
    return messages

In [ ]:
reset_demo()

options = ClaudeAgentOptions(
    allowed_tools=["Read", "Edit", "Bash", "Glob"],  # auto-approved tools
    permission_mode="acceptEdits",                   # auto-approve file edits
    cwd=str(DEMO_DIR),                               # the agent works only in this folder
)

messages = await trace(
    "Run the unit tests with `python -m unittest`, fix the bug in calculator.py, "
    "then run the tests again to confirm.",
    options,
)

In [ ]:
run_tests()   # verify independently of the agent
print((DEMO_DIR / "calculator.py").read_text())

**Read the trace like a story.** A typical run: turn 1 runs the tests and sees one failure (`test_add`), turn 2 reads the code, turn 3 edits the file and re-runs the tests, then a final text-only message ends the loop. The agent chose that path itself, adjusting after each result. That is what "agentic" means.

Quick tally of what was yielded:

In [ ]:
from collections import Counter

print(Counter(type(m).__name__ for m in messages))

## 3. Tools: what the agent can *do*

Without tools Claude can only answer with text. The SDK includes the same built-in tools as Claude Code:

| Category | Tools | What they do |
|---|---|---|
| File operations | `Read`, `Edit`, `Write` | Read, modify, create files |
| Search | `Glob`, `Grep` | Find files by pattern, search contents |
| Execution | `Bash` | Run shell commands, scripts, git |
| Web | `WebSearch`, `WebFetch` | Search and fetch pages |
| Discovery | `ToolSearch` | Load tools on demand instead of preloading all |
| Orchestration | `Agent`, `Skill`, `AskUserQuestion`, `TaskCreate`, `TaskUpdate` | Subagents, skills, asking the user, task tracking |

You can add more with **MCP servers** (databases, browsers, APIs) and **custom tools**.

### Parallel vs sequential
When Claude requests several tools in one turn, **read-only** tools (`Read`, `Glob`, `Grep`, read-only MCP tools) may run **concurrently**. Tools that **change state** (`Edit`, `Write`, `Bash`) run **one after another** to avoid conflicts. Custom tools are sequential unless you mark them with `readOnlyHint`.

### Controlling permissions: three options that work together

| Option | Effect |
|---|---|
| `allowed_tools` | **Auto-approves** the listed tools (no prompt). Unlisted tools are still *available*, but calls needing approval fall through to the permission mode |
| `disallowed_tools` | **Blocks** the listed tools, whatever else is set |
| `permission_mode` | How much human oversight you want overall |

When a tool is denied, **Claude receives a rejection message as the tool result** and usually tries another approach or explains it could not proceed. The loop does not crash.

You can also scope a tool with a rule, such as `"Bash(python *)"`, to allow only certain commands.

#### Experiment A: Bash is not allowed
With no approval callback and the default permission mode, a call that needs approval is **denied**. Watch the agent react, and look at `permission_denials` on the result.

In [ ]:
reset_demo()

opts_no_bash = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob"],      # Bash is NOT pre-approved
    permission_mode="default",
    cwd=str(DEMO_DIR),
)
msgs = await trace("Run `python -m unittest` and tell me what it printed.", opts_no_bash)

result = next(m for m in msgs if isinstance(m, ResultMessage))
print("\npermission_denials:", result.permission_denials)

#### Experiment B: allowed *and* disallowed, so disallowed wins
`disallowed_tools` blocks a tool regardless of every other setting.

In [ ]:
opts_blocked = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Bash"],
    disallowed_tools=["Bash"],           # wins over allowed_tools
    cwd=str(DEMO_DIR),
)
msgs = await trace("Run `python -m unittest` and tell me what it printed.", opts_blocked)

#### Experiment C: a scoped rule
Allow only `python ...` commands. The first command should run; the second (`ls`) is not covered by the rule, so it is denied. (Exact behavior can vary a little by model and version.)

In [ ]:
opts_scoped = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Bash(python *)"],
    cwd=str(DEMO_DIR),
)
msgs = await trace(
    "First run `python -m unittest`. Then run `ls -la`. Report what happened to each.",
    opts_scoped,
)

## 4. Controlling how long and how deep the loop runs

All of these are fields on `ClaudeAgentOptions`.

| Option | Controls | Default |
|---|---|---|
| `max_turns` | Max tool-use round trips (counts **tool-use turns only**) | no limit |
| `max_budget_usd` | Estimated spend at which the loop stops | no limit |
| `effort` | How much reasoning Claude applies: `low`, `medium`, `high`, `xhigh`, `max` | resolved by Claude Code |

When a limit is hit you get a `ResultMessage` whose `subtype` is `error_max_turns` or `error_max_budget_usd`, and a single-shot `query()` then **raises**. That is why the trace helper wraps the loop in `try/except`.

> Without limits the loop runs until Claude decides it is done. Fine for well-scoped tasks, risky for open-ended ones ("improve this codebase"). **Setting a budget is a good default for production agents.**

#### Experiment D: cap the turns
The task needs more than one tool turn, so `max_turns=1` stops it early.

In [ ]:
reset_demo()

opts_limited = ClaudeAgentOptions(
    allowed_tools=["Read", "Edit", "Bash", "Glob"],
    permission_mode="acceptEdits",
    cwd=str(DEMO_DIR),
    max_turns=1,
)
msgs = await trace("Run the tests, fix the bug in calculator.py, and run the tests again.", opts_limited)
run_tests()   # the fix probably did not land: the loop was cut off

#### Experiment E: cap the budget
Cost is compared with the cap **after each model response arrives**, so the response that crosses the cap still completes. Spend can pass the cap by up to one response. Leave headroom.

In [ ]:
reset_demo()

opts_budget = ClaudeAgentOptions(
    allowed_tools=["Read", "Edit", "Bash", "Glob"],
    permission_mode="acceptEdits",
    cwd=str(DEMO_DIR),
    max_budget_usd=0.01,     # deliberately tiny
)
msgs = await trace("Run the tests, fix the bug in calculator.py, and run the tests again.", opts_budget)

#### Experiment F: effort level
`effort` trades reasoning depth for latency and cost. Use `low` for simple lookups. Not every model supports it, so this cell catches errors.

In [ ]:
for level in ("low", "high"):
    print(f"\n===== effort={level} =====")
    try:
        await trace(
            "List the files in this folder and say which one contains the tests.",
            ClaudeAgentOptions(allowed_tools=["Glob", "Read"], cwd=str(DEMO_DIR), effort=level),
        )
    except Exception as error:
        print("This model/version rejected effort:", short(error, 150))

## 5. Permission modes

`permission_mode` sets the overall level of oversight:

| Mode | Behavior | Typical use |
|---|---|---|
| `default` | Calls needing approval go to your `canUseTool` callback; no callback = deny | Interactive apps with an approval UI |
| `acceptEdits` | Auto-approves file edits and common filesystem commands (`mkdir`, `touch`, `mv`, `cp`...) | Prototyping, isolated folders |
| `plan` | Claude explores and plans; **edits are never auto-approved** | Code review, "propose but don't touch" |
| `dontAsk` | Never prompts: pre-approved tools run, everything else is denied | Headless agents with a fixed tool surface |
| `auto` | A model classifier approves or blocks each risky action | Autonomous agents that still want guardrails |
| `bypassPermissions` | Runs everything allowed without asking | **Only** isolated CI or containers |

#### Experiment G: plan mode
Ask for a fix in `plan` mode. The agent should **describe** the change and leave the file alone.

In [ ]:
reset_demo()

opts_plan = ClaudeAgentOptions(
    allowed_tools=["Read", "Edit", "Glob"],
    permission_mode="plan",
    cwd=str(DEMO_DIR),
)
msgs = await trace("Fix the bug in calculator.py.", opts_plan)

print("\nFile still buggy?", (DEMO_DIR / "calculator.py").read_text() == BUGGY_CALCULATOR)

## 6. The context window

The **context window** is everything Claude can see in a session. It does **not** reset between turns. It accumulates:

| Source | When it loads | Impact |
|---|---|---|
| System prompt | Every request | Small fixed cost |
| `CLAUDE.md` files | Session start (via `setting_sources`) | In every request, but prompt-cached |
| Tool definitions | Every request (MCP schemas are deferred by default) | Each tool costs context |
| Conversation history | Grows each turn | Prompts, replies, tool inputs **and outputs** |
| Skill descriptions | Session start | Short; full text loads only when used |

Big tool outputs (a huge file, verbose command output) can use thousands of tokens in one turn.

### Automatic compaction
Near the limit, the SDK **summarizes older history** to free space, keeping recent exchanges and key decisions. You get a `SystemMessage` with `subtype="compact_boundary"`. Because early instructions may be summarized away:

- Put **persistent rules in `CLAUDE.md`**, which is re-injected on every request, not only in the first prompt.
- You can add a "summary instructions" section to `CLAUDE.md` telling the compactor what to preserve.
- A `PreCompact` hook can archive the full transcript first. Sending `/compact` as a prompt triggers it manually.

### Keeping context lean
- Use **subagents** for side tasks: only their final answer returns to the parent.
- Be selective with tools; every definition costs space.
- Use `effort="low"` for routine work.

Here is how to spot compaction in a stream:

In [ ]:
def saw_compaction(messages):
    return any(isinstance(m, SystemMessage) and m.subtype == "compact_boundary" for m in messages)

print("Compaction happened in the earlier run?", saw_compaction(messages))

## 7. Handle the result properly

The last message is a `ResultMessage`. Its `subtype` is the primary way to know **how the loop ended**:

| `subtype` | What happened | `result` text available? |
|---|---|:-:|
| `success` | Finished normally | yes |
| `error_max_turns` | Hit `max_turns` | no |
| `error_max_budget_usd` | Hit `max_budget_usd` | no |
| `error_during_execution` | Interrupted (cancelled request, crash...) | no |
| `error_max_structured_output_retries` | No valid structured output within retry limit | no |

Every result also carries `total_cost_usd`, `usage`, `num_turns` and `session_id`, so you can track cost and **resume** even after an error. Two cautions: in Python `total_cost_usd` / `usage` may be `None`, and `usage` covers only the main loop (use `model_usage` for subagent totals). `stop_reason` tells you why the model stopped (`end_turn`, `max_tokens`, `refusal`...).

The pattern below is the one to copy into real code:

In [ ]:
async def run_agent(prompt, options):
    session_id = None
    try:
        async for message in query(prompt=prompt, options=options):
            if isinstance(message, ResultMessage):
                session_id = message.session_id           # save for resume
                if message.subtype == "success":
                    print("Done:", short(message.result, 300))
                elif message.subtype == "error_max_turns":
                    print(f"Hit the turn limit. Resume session {session_id} with a higher limit.")
                elif message.subtype == "error_max_budget_usd":
                    print("Hit the budget limit.")
                else:
                    print("Stopped:", message.subtype)
                if message.total_cost_usd is not None:
                    print(f"Cost: ${message.total_cost_usd:.4f}")
                print("stop_reason:", message.stop_reason, "| turns:", message.num_turns)
    except Exception as error:
        print("Session ended with an error:", short(error, 150))
    return session_id

reset_demo()
sid = await run_agent(
    "Run the tests, fix the bug in calculator.py, and run the tests again.",
    ClaudeAgentOptions(
        allowed_tools=["Read", "Edit", "Bash", "Glob"],
        permission_mode="acceptEdits",
        cwd=str(DEMO_DIR),
        max_turns=30,        # safety net against runaway sessions
        effort="medium",
    ),
)
print("session id:", sid)

## 8. Hooks: your code, inside the loop

**Hooks** are callbacks that fire at specific points: before a tool runs, after it returns, when the agent stops, before compaction... They run **in your own process**, not inside the context window, so they cost no context.

| Hook | When | Common uses |
|---|---|---|
| `PreToolUse` | Before a tool executes | Validate inputs, **block** dangerous commands |
| `PostToolUse` | After a tool returns | Audit outputs, side effects |
| `UserPromptSubmit` | When a prompt is sent | Inject extra context |
| `Stop` | When the agent finishes | Validate result, save state |
| `SubagentStart` / `SubagentStop` | Subagent lifecycle | Aggregate parallel work |
| `PreCompact` | Before compaction | Archive the transcript |

A callback returns `{}` to **allow**, or a `hookSpecificOutput` with `permissionDecision: "deny"` to **block**. The deny reason is sent to Claude as the tool result, so it adapts.

#### Experiment H: a logging hook (observes, never blocks)

In [ ]:
tool_log = []

async def log_tool(input_data, tool_use_id, context):
    tool_log.append((input_data.get("tool_name"), short(input_data.get("tool_input"), 70)))
    return {}   # empty dict = allow

reset_demo()
await trace(
    "Run the tests, fix the bug in calculator.py, and run the tests again.",
    ClaudeAgentOptions(
        allowed_tools=["Read", "Edit", "Bash", "Glob"],
        permission_mode="acceptEdits",
        cwd=str(DEMO_DIR),
        hooks={"PreToolUse": [HookMatcher(hooks=[log_tool])]},
    ),
)

print("\nWhat the hook saw (in order):")
for name, args in tool_log:
    print(f"  {name:6} {args}")

#### Experiment I: a hook that blocks edits
The hook below rejects every `Edit`. The agent still gets a tool result (the rejection reason) and must deal with it.

In [ ]:
async def block_edits(input_data, tool_use_id, context):
    return {
        "hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": "Editing is frozen for this demo.",
        }
    }

reset_demo()
await trace(
    "Fix the bug in calculator.py.",
    ClaudeAgentOptions(
        allowed_tools=["Read", "Edit", "Glob"],
        permission_mode="acceptEdits",
        cwd=str(DEMO_DIR),
        hooks={"PreToolUse": [HookMatcher(matcher="Edit", hooks=[block_edits])]},
    ),
)
print("\nFile still buggy?", (DEMO_DIR / "calculator.py").read_text() == BUGGY_CALCULATOR)

## 9. Recap

- The loop is **evaluate → tool calls → results → repeat**, ending when Claude replies with no tool calls. One cycle is a **turn**.
- You observe it as a stream of **five message types**; the last is always a `ResultMessage`.
- **Tools** give the agent power; `allowed_tools`, `disallowed_tools` and `permission_mode` decide what is allowed to run. Denials come back to Claude as tool results.
- **`max_turns`, `max_budget_usd`, `effort`** bound the loop. Always set limits for production.
- **Context accumulates**; compaction summarizes old history, so put lasting rules in `CLAUDE.md`.
- Always check `ResultMessage.subtype` before reading `.result`, and save `session_id` to resume later.
- **Hooks** let you observe, block, or modify behavior from your own code.

### Try it yourself
1. Add `"Write"` to the allowed tools and ask the agent to create `README.md` documenting `calculator.py`. Trace the turns.
2. Set `include_partial_messages=True` and watch `StreamEvent` messages arrive.
3. Write a `PostToolUse` hook that records how long each tool call took.
4. Next notebook: **`agent-sdk-claude-code-features-explained.ipynb`** (settings, CLAUDE.md, skills, hooks from files).